# 01 - Data Inspection & Cleaning

**Phase 1 scope:** data quality audit, missing values, duplicates, inconsistent values, key validation, and reproducible cleaning.

This notebook inspects the cleaned outputs generated from the original Instacart dataset. The original Kaggle CSV files must be downloaded separately and placed in `data/raw/`; they are not bundled with this repository.

**No modeling is performed.**

**Data mode:** `real` uses a reproducible sample of observed Instacart transactions. Run the Phase 1 pipeline before executing this notebook.

In [6]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

## Analysis of a reproducible sample of 30,000 prior orders

In [7]:
DATA_MODE = 'real'

print('DATA MODE:', DATA_MODE.upper(), '|', 'REAL INSTACART SAMPLE' if DATA_MODE == 'real' else 'SYNTHETIC DEMO')

DATA MODE: REAL | REAL INSTACART SAMPLE


## Load source tables and inspect schema
We sample prior **orders** reproducibly before joining the potentially huge line-items file. This avoids silently treating a random selection of line items as complete baskets.

In [8]:
import pandas as pd
import json
from IPython.display import display

processed = ROOT / 'data/processed'

report = json.loads(
    (processed / 'cleaning_report.json').read_text()
)

print('Selected prior orders:', report['selected_prior_orders'])
print('Maximum orders requested:', report['max_orders_requested'])
print('Random seed:', report['random_seed'])

print('Sampling method:', report['sampling_method'])

for name, audit_data in report['audit_before'].items():
    print(
        name,
        'rows:', audit_data['rows'],
        'missing:', audit_data['missing_by_column'],
        'duplicates:', audit_data['exact_duplicates']
    )

Selected prior orders: 30000
Maximum orders requested: 30000
Random seed: 42
Sampling method: Deterministic random sample of prior order IDs using seed 42. Complete product lines for selected orders are filtered in chunks. This is not a chronological split.
orders rows: 30000 missing: {'order_id': 0, 'user_id': 0, 'eval_set': 0, 'order_number': 0, 'order_dow': 0, 'order_hour_of_day': 0, 'days_since_prior_order': 1894} duplicates: 0
order_products rows: 302305 missing: {'order_id': 0, 'product_id': 0, 'add_to_cart_order': 0, 'reordered': 0} duplicates: 0
products rows: 49688 missing: {'product_id': 0, 'product_name': 0, 'aisle_id': 0, 'department_id': 0} duplicates: 0
aisles rows: 134 missing: {'aisle_id': 0, 'aisle': 0} duplicates: 0
departments rows: 21 missing: {'department_id': 0, 'department': 0} duplicates: 0


## Review documented cleaning
- Remove exact and key-based duplicates; remove rows without critical IDs.
- Validate time and item-position ranges and catalog foreign keys.
- Keep first-order `days_since_prior_order` nulls (structurally expected).
- Do not remove valid large baskets as “outliers” automatically.
- Generate `cleaning_report.json` containing before/after audit counts.

In [9]:
# Cleaning is implemented in scripts/pipeline.py.
# Run the pipeline from the Phase1 directory before executing this notebook:
# python -m scripts.run_phase1 --max-orders 30000

print('Cleaning decisions:', report['cleaning'])
print('After cleaning:', report['after'])

outputs = {}

for name in [
    'orders',
    'product_catalog',
    'order_products_clean',
    'baskets'
]:
    outputs[name] = pd.read_csv(
        processed / f'{name}.csv',
        nrows=10
    )

Cleaning decisions: {'removed_duplicate_or_invalid_key_rows': {'orders': 0, 'order_products': 0, 'products': 0, 'aisles': 0, 'departments': 0, 'orders_invalid_user_id': 0}, 'invalid_order_time_rows_removed': 0, 'invalid_line_flags_or_positions_removed': 0, 'invalid_catalog_rows_removed': 0, 'first_order_missing_days_since_prior_order': 'Kept as NA because it represents expected structural missingness.', 'outlier_policy': 'Invalid order times and cart positions are excluded. Genuine large baskets are retained. Days since prior order must be within 0 to 30.', 'scaling': 'Not applied: product IDs are categorical identifiers. Apriori and FP-Growth use item presence in baskets, not continuous product ID values.'}
After cleaning: {'orders': 30000, 'order_product_rows': 302305, 'products_catalog': 49688, 'nonempty_baskets': 30000}


## Inspect the cleaned outputs
These tables support future basket-based mining, but neither recommendations nor rule models are trained in this phase.

In [10]:
display(outputs['product_catalog'].head());display(outputs['baskets'].head())

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


,order_id,user_id,basket_size,product_ids,product_names,order_dow,order_hour_of_day
0,136,184355,4,11005|18465|29926|47766,Organic Cream Cheese Bar | Organic Grade A Fre...,6,10
1,376,121858,7,890|2716|8277|16462|35851|45210|47029,Organic Diced Tomatoes | Organic Greek Lowfat ...,5,16
2,522,35250,20,1862|2344|10251|13292|17706|24184|24852|25340|...,Tuna Fillets in Olive Oil | Organic SprouTofu ...,1,15
3,526,17900,18,1752|1831|1940|9387|12191|12302|22489|23085|27...,Mango Sorbet | Peach-Pear Sparkling Water | Or...,0,23
4,552,18926,4,4091|5268|33452|37011,Kosher Whole Dill Pickles | Bagel Bites Three ...,4,13
